In [1]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

In [2]:
df = pd.read_excel("ExampleStep3Output.xlsx") 
df

,Author 1: Relevant Article? (Yes/No),Author 2: Relevant Article? (Yes/No),date_published,title,keywords,abstract,PMID,authors,journal,citation,URL,Downloaded,Text,category
0,No,No,1948 Sep-Oct,[Injection of the sympathetic superior cervica...,['*ANESTHESIA AND ANESTHETICS/spinal--complica...,No abstract available,18893366,"['MOYSON F', 'LAMBIOTTE C']",Revue de chirurgie,"MOYSON, F., LAMBIOTTE, C. (1948 Sep-Oct). [Inj...",Not available,False,Text not available,'Other Conditions'
1,No,No,2019 May,Intracranial hypertension in a patient with Cl...,"['Acetazolamide/*administration & dosage', 'An...",No abstract available,30995704,"['Chia XX', 'Cordato D', 'Venkat A', 'Spicer ST']","Nephrology (Carlton, Vic.)","Chia, XX., Cordato, D., Venkat, A., Spicer, ST...",https://libkey.io/libraries/731/articles/29959...,False,Text not available,'Other Conditions'
2,No,No,2024 Jan,Case 325.,"['Humans', 'Female', 'Aged', '*Deglutition Dis...",A 76-year-old woman with a history of rheumato...,38289217,"['Suthar PP', 'Nagarajan M', 'Bhabad S']",Radiology,"Suthar, PP., Nagarajan, M., Bhabad, S. (2024 J...",https://libkey.io/libraries/731/articles/60391...,False,Text not available,'Neurological Disorders'
3,No,No,2020 Feb,Postdural Puncture Headache and Acute Subdural...,"['Acute subdural haematoma', ""Sjogren's syndro...",Although spontaneous intracranial hypotension ...,32076685,"['Yildirim HU', 'Bakir M', 'Atici SR']",Turkish journal of anaesthesiology and reanima...,"Yildirim, HU., Bakir, M., Atici, SR. (2020 Feb...",https://europepmc.org/backend/ptpmcrender.fcgi...,True,Doi: 10.5152/TJAR.2019.73444 Case Report\nAlgo...,'Neurological Disorders'
4,No,No,2014 Nov,An unusual cause of chronic headaches: question.,"['Arthritis, Rheumatoid/*complications/patholo...",No abstract available,25587612,"['Zuzuarregui JR', 'Lin J', 'Otis JA']",Journal of clinical neuroscience : official jo...,"Zuzuarregui, JR., Lin, J., Otis, JA. (2014 Nov...",https://libkey.io/libraries/731/articles/52358...,False,Text not available,'Other Conditions'
5,No,No,2015 May,Marfan syndrome presenting with postpartum aor...,"['Adult', 'Aortic Dissection/complications/*di...",No abstract available,25790894,"['Humphrey V', 'Falzon D', 'Clark V']",International journal of obstetric anesthesia,"Humphrey, V., Falzon, D., Clark, V. (2015 May)...",https://libkey.io/libraries/731/articles/53923...,False,Text not available,'Other Conditions'
6,No,No,1999 Jan-Feb,Progressive systemic sclerosis: intrathecal pa...,"['Aged', 'Analgesics, Opioid/*administration &...","BACKGROUND AND OBJECTIVES: At present, there i...",9952101,"['Lundborg CN', 'Nitescu PV', 'Appelgren LK', ...",Regional anesthesia and pain medicine,"Lundborg, CN., Nitescu, PV., Appelgren, LK., C...",Not available,False,Text not available,'Autoimmune Disorders'
7,No,No,2008 Aug,Intracranial hypertension in 2 children with m...,"['Amino Acid Substitution/genetics', 'Brain/pa...",Two unrelated children with Marfan syndrome pr...,18354149,"['Hilhorst-Hofstee Y', 'Kroft LJ', 'Pals G', '...",Journal of child neurology,"Hilhorst-Hofstee, Y., Kroft, LJ., Pals, G., va...",https://libkey.io/libraries/731/articles/14884...,False,Text not available,'Neurological Disorders'
8,No,No,2016 Jun,[Safe and Effective Analgesia with Bilateral C...,"['Anesthesia, Epidural', 'Aortic Aneurysm, Abd...",A patient with Marfan syndrome underwent emerg...,27483660,"['Katakura Y', 'Sakurai A', 'Endo M', 'Hamada ...",Masui. The Japanese journal of anesthesiology,"Katakura, Y., Sakurai, A., Endo, M., Hamada, T...",Not available,False,Text not available,'Other Conditions'
9,No,No,2020 Jan,Parameters Related to Lumbar Puncture Do not A...,"['Headache', 'Lumbar puncture', 'Post-dural pu...",BACKGROUND: Post-dural puncture headache (PDPH...,31562971,"['Ljubisavljevic S', 'Trajkovic JZ', 'Ignjatov...",World neurosurgery,"Ljubisavljevic, S., Trajkovic, JZ., Ignjatovic...",https://libkey.io/libraries/731/articles/34577...,False,Text not available,'Neurological Diso

In [3]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [4]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
# from langchain_text_splitters import CharacterTextSplitter

CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2023-06-01-preview",
    azure_deployment="ChatGPT432k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-4-32k",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Use APA-style in-text citations throughout the paragraph. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragraph summarizing the overall contents, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [ ]:
# use abtract when text is not available.
df['Text'] = df.apply(lambda row: row['abstract'] if row['Text'] == 'Text not available' else row['Text'], axis=1)

output = []
for current_category in categories:
    filtered_rows = df[(df['category'] == current_category)]

    article_summaries = []
    for _, row in filtered_rows.iterrows():
        text_spilt = text_splitter.create_documents([row.Text])
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content="APA Citation: " + row.citation + "\n\nArticle text:\n\n" +text_spilt,
        ).to_messages())

        summary_to_keep = f"APA Citation: {row.citation}\n\n Summary: {result.content}\n\n --- "
        print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    
    chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
    
    print(result.content)
    output.append(result.content)
    print("************")
    

In [ ]:
print(output)

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

In [ ]:
!pip install --quiet langchain_experimental langchain_openai



In [ ]:
text_splitter = SemanticChunker(OpenAIEmbeddings())

In [9]:
%pip install -qU langchain-text-splitters


[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [ ]:
prompt_template = """Write a concise summary of the following:
{text}
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

refine_template = (
    "Your job is to produce a final summary\n"
    "We have provided an existing summary up to a certain point: {existing_answer}\n"
    "We have the opportunity to refine the existing summary"
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{text}\n"
    "------------\n"
    "Given the new context, refine the original summary in Italian"
    "If the context isn't useful, return the original summary."
)
refine_prompt = PromptTemplate.from_template(refine_template)
chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
    input_key="input_documents",
    output_key="output_text",
)
result = chain({"input_documents": split_docs}, return_only_outputs=True)

In [7]:
def summarize_article_in_chunks(article_text):
    # Splitting the article text into manageable chunks
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=25000, chunk_overlap=1000)
    texts = text_splitter.create_documents([article_text])
 
    # Template for the initial summarization of the first chunk
    initial_summary_prompt = """I am working on a scoping review to address a specific question.
    I need to summarize this journal article, focusing on the given question and the article's category.
    Here is the first chunk of the journal article:
 
    {text}
    """
 
    # Template for refining the summary with each subsequent chunk
    refine_summary_prompt = """Based on the existing summary and the specific question, refine the summary with the information from the next chunk of the article.
    Current summary:
 
    {existing_summary}
 
    Next chunk of the article:
    ------------
    {text}
    ------------
    """
 
    # Create the initial summary for the first chunk
    summary = CHAT.invoke(initial_summary_prompt.format(text=texts[0]))
 
    # Iteratively refine the summary with each subsequent chunk
    for text_chunk in texts[1:]:
        summary = CHAT.invoke(refine_summary_prompt.format(existing_summary=summary, text=text_chunk))
 
    return summary
 
# Usage of the function within the larger script
for current_category in categories:
    filtered_rows = df[df['category'] == current_category]
    output = []
 
    for _, row in filtered_rows.iterrows():
        article_summary = summarize_article_in_chunks(row.Text)
        formatted_summary = f"APA Citation: {row.citation}\n\n Summary: {article_summary}\n\n --- "
        output.append(formatted_summary)
        
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
    question=user_question,
    category=current_category,
    content=output
    ).to_messages())
    print(current_category)
    print(result.content)


'Other Conditions'
Unfortunately, due to the unavailability of the full text of the articles, it is not possible to provide a comprehensive summary addressing the question of whether a diagnosis of a connective tissue disease contributes to a post-dural spinal puncture headache. The articles cited include works by Moyson and Lambiotte (1948), Chia et al. (2019), Zuzuarregui et al. (2014), Humphrey et al. (2015), Katakura et al. (2016), and Cestari et al. (2018). However, without the full text, it is impossible to extract relevant information or draw conclusions related to the research question.
'Neurological Disorders'
The literature suggests a potential link between connective tissue diseases and the occurrence of post-dural spinal puncture headaches. Yildirim, Bakir, and Atici (2020) reported a case of a patient with Sjögren’s Syndrome, a connective tissue disease, who developed postdural puncture headache and acute subdural haematoma after spinal anaesthesia. The authors suggested t

In [15]:
print(len(output))

17
